# Vídeos para YouTube desde Colab

Abrir: https://colab.research.google.com/github/DreiSup/spanishnav.al-podcast/blob/claude/quirky-dirac-x18c3n/video/video.ipynb

Hace el `.mp4` de uno o varios episodios sin pasar por tu ordenador: **portada fija + master**, y lo deja en Drive, en
la carpeta `video/` de cada episodio (`docs/sop-publicacion.md`, apartado 5 bis).

**Entorno: CPU** (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU*). No hace falta GPU.

**Uso:** pon los episodios en la celda 1 y *Entorno de ejecución → Ejecutar todas*. Drive pedirá permiso la primera vez.

Para cada episodio:

1. Comprueba en el repo que el master está **escuchado y dado por bueno** (`metadata.es.json` → `escucha`); si no,
   lo salta (se puede desactivar con `SOLO_ESCUCHADOS = False`).
2. Copia el master de Drive al disco de Colab y comprueba que se decodifica sin errores.
3. Hace el vídeo: 1920×1080, H.264, 25 fps; audio AAC-LC estéreo a 48 kHz y 192 kbps (lo que recomienda YouTube).
4. Comprueba el resultado: formato, duraciones, que vídeo y audio se decodifican sin errores y que **el audio del vídeo
   es el del master** (lo compara en 12 tramos repartidos por todo el episodio).
5. Lo copia a Drive.

Si un episodio falla, lo apunta y **sigue con el siguiente**. Al final, un resumen y 30 s de cada vídeo para
escucharlos aquí. No juzgues el sonido por la vista previa de Drive: recién subido, Drive lo reproduce a calidad baja.

Tarda aproximadamente un cuarto de la duración de cada episodio (7 min de audio, ~2 min).

In [ ]:
# 1. Qué episodios. Ejemplos: "004" · "006-037" · "006, 008, 010-012" · un extra sin número: "2019/extra-on-meditation"
EPISODIOS = "006-037"
SOLO_ESCUCHADOS = True   # True = solo los que en el repo están escuchados y dados por buenos
SOBRESCRIBIR = False     # True = rehace el vídeo aunque ya exista en Drive (si no, lo salta)

In [ ]:
# 2. Preparar: Drive, ffmpeg, la portada y las funciones
import subprocess, shutil, time, json, re, urllib.request
from pathlib import Path
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

RAIZ = Path("/content/drive/MyDrive/naval-podcast/transcripts")
REPO_RAW = "https://raw.githubusercontent.com/DreiSup/spanishnav.al-podcast/claude/quirky-dirac-x18c3n"
TRABAJO = Path("/content/video_trabajo")
shutil.rmtree(TRABAJO, ignore_errors=True)
TRABAJO.mkdir()
PORTADA = TRABAJO / "portada-youtube.jpg"
subprocess.run(["curl", "-sSfL", "-o", str(PORTADA), "https://spanishnav.al/portada-youtube.jpg"], check=True)
VF = "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2,format=yuv420p"


class Salto(Exception):
    """El episodio no se hace, pero no es un error (sin escuchar, ya hecho…)."""


def lista_de_episodios(texto):
    """'006-037', '006, 008, 010-012', '4' o '2019/extra-…' → ['006', '007', …]."""
    salida = []
    for parte in str(texto).split(","):
        parte = parte.strip()
        if not parte:
            continue
        m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", parte)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            assert a <= b, f"Rango al revés: {parte}"
            salida += [f"{n:03d}" for n in range(a, b + 1)]
        elif parte.isdigit():
            salida.append(f"{int(parte):03d}")
        else:
            salida.append(parte)               # extra sin número: <año>/<carpeta>
    return list(dict.fromkeys(salida))


def sonda(ruta, campos, flujo=None):
    cmd = ["ffprobe", "-v", "error"] + (["-select_streams", flujo] if flujo else []) + \
          ["-show_entries", campos, "-of", "default=nw=1", str(ruta)]
    salida = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
    return dict(l.split("=", 1) for l in salida.strip().splitlines() if "=" in l)


def errores_al_decodificar(ruta, mapa):
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-map", mapa, "-f", "null", "-"],
                       capture_output=True, text=True)
    return r.stderr.strip() or ("ffmpeg terminó con código %d" % r.returncode if r.returncode else "")


def buscar(ep):
    """Carpeta del episodio en Drive y su master."""
    if ep.isdigit():
        carpetas = sorted(p for p in RAIZ.glob(f"20*/{ep} *") if p.is_dir())
        if not carpetas:
            raise Salto("no hay carpeta en Drive")
        assert len(carpetas) == 1, f"hay {len(carpetas)} carpetas '{ep} …': {[c.name for c in carpetas]}"
        carpeta = carpetas[0]
        masters = sorted((carpeta / "audio").glob(f"{ep}-*.mp3"))
    else:
        carpeta = RAIZ / ep
        masters = sorted((carpeta / "audio").glob(f"{carpeta.name}.mp3"))
    if not masters:
        raise Salto("todavía no tiene master")
    assert len(masters) == 1, f"hay {len(masters)} masters: {[m.name for m in masters]}"
    return carpeta, masters[0]


def escuchado(anio, nombre):
    try:
        with urllib.request.urlopen(f"{REPO_RAW}/episodios/{anio}/{nombre}/metadata.es.json", timeout=30) as r:
            meta = json.load(r)
    except Exception as e:
        raise Salto(f"no encuentro su metadata.es.json en el repo ({e})")
    esc = meta.get("escucha") or {}
    return bool(esc.get("escuchado")) and esc.get("resultado") == "bueno"


def parecido(master, video, dur):
    """SNR (dB) del audio del vídeo frente al master en 12 tramos de 10 s. AAC bueno: 20–35 dB; un corte o un salto,
    cerca de 0 desde ese punto; un audio distorsionado, menos de 10 (probado el 6-nov con vídeos estropeados)."""
    def tramo(ruta, desde):
        crudo = subprocess.run(["ffmpeg", "-v", "error", "-ss", f"{desde:.3f}", "-t", "10", "-i", str(ruta),
                                "-map", "0:a:0", "-ac", "1", "-ar", "48000", "-f", "f32le", "-"],
                               capture_output=True, check=True).stdout
        return np.frombuffer(crudo, dtype=np.float32)
    snr = []
    for desde in np.linspace(1, max(1, dur - 12), 12):
        x, y = tramo(master, desde), tramo(video, desde)
        n = min(len(x), len(y))
        x, y = x[:n].astype(np.float64), y[:n].astype(np.float64)
        if n < 48000 or np.sqrt(np.mean(x ** 2)) < 1e-3:
            continue                                   # tramo demasiado corto o en silencio
        corr = np.fft.irfft(np.fft.rfft(x, 2 * n) * np.conj(np.fft.rfft(y, 2 * n)))
        lag = int(np.argmax(np.concatenate([corr[-2400:], corr[:2401]]))) - 2400   # desfase de ±50 ms como mucho
        xs, ys = (x[lag:], y[:n - lag]) if lag >= 0 else (x[:n + lag], y[-lag:])
        g = np.dot(xs, ys) / np.dot(ys, ys)
        snr.append(10 * np.log10(np.sum(xs ** 2) / np.sum((xs - g * ys) ** 2)))
    return snr


def hacer_video(ep):
    for f in list(TRABAJO.glob("*.mp3")) + list(TRABAJO.glob("*.mp4")):
        f.unlink()                                     # restos de un episodio anterior que falló
    carpeta, master_drive = buscar(ep)
    anio, nombre = carpeta.parent.name, master_drive.stem
    destino = carpeta / "video" / f"{nombre}.mp4"
    if destino.exists() and not SOBRESCRIBIR:
        raise Salto("ya tiene vídeo en Drive (SOBRESCRIBIR = False)")
    if SOLO_ESCUCHADOS and not escuchado(anio, nombre):
        raise Salto("no está escuchado y dado por bueno en el repo")

    master = TRABAJO / master_drive.name
    shutil.copyfile(master_drive, master)
    assert master.stat().st_size == master_drive.stat().st_size, "la copia del master no tiene el tamaño del original"
    dur = float(sonda(master, "format=duration")["duration"])
    fallo = errores_al_decodificar(master, "0:a:0")
    assert not fallo, f"el master de Drive tiene errores al decodificarlo: {fallo[:500]}"

    salida = TRABAJO / f"{nombre}.mp4"
    r = subprocess.run([
        "ffmpeg", "-v", "error", "-y",
        "-loop", "1", "-framerate", "25", "-i", str(PORTADA),
        "-i", str(master),
        "-map", "0:v:0", "-map", "1:a:0", "-t", f"{dur:.3f}",
        "-vf", VF, "-r", "25", "-c:v", "libx264", "-preset", "veryfast", "-tune", "stillimage", "-crf", "20",
        "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
        "-movflags", "+faststart", str(salida)], capture_output=True, text=True)
    assert r.returncode == 0 and salida.is_file(), f"ffmpeg falló: {r.stderr[-1500:]}"

    v = sonda(salida, "stream=codec_name,width,height,pix_fmt,duration", "v:0")
    a = sonda(salida, "stream=codec_name,sample_rate,channels,duration", "a:0")
    assert (v["codec_name"], v["width"], v["height"], v["pix_fmt"]) == ("h264", "1920", "1080", "yuv420p"), f"vídeo: {v}"
    assert (a["codec_name"], a["sample_rate"], a["channels"]) == ("aac", "48000", "2"), f"audio: {a}"
    assert abs(float(v["duration"]) - dur) <= 0.5 and abs(float(a["duration"]) - dur) <= 0.5, \
        f"duraciones: master {dur:.2f} s, vídeo {v['duration']}, audio {a['duration']}"
    for mapa in ("0:v:0", "0:a:0"):
        fallo = errores_al_decodificar(salida, mapa)
        assert not fallo, f"el .mp4 tiene errores al decodificar {mapa}: {fallo[:500]}"
    snr = parecido(master, salida, dur)
    assert len(snr) >= 6, "demasiados tramos en silencio para comparar"
    assert np.median(snr) >= 20 and min(snr) >= 12, \
        f"el audio del vídeo no se parece al master (mediana {np.median(snr):.1f} dB, mínimo {min(snr):.1f} dB)"

    destino.parent.mkdir(exist_ok=True)
    temporal = destino.with_suffix(".mp4.parcial")
    shutil.copyfile(salida, temporal)
    temporal.replace(destino)
    assert destino.stat().st_size == salida.stat().st_size, "la copia en Drive no tiene el tamaño del vídeo"

    muestra = TRABAJO / f"{nombre}_muestra.wav"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", str(min(60, max(0, dur - 30))), "-t", "30", "-i", str(salida),
                    "-map", "0:a:0", str(muestra)], check=True)
    master.unlink()
    salida.unlink()                                    # el disco de Colab es limitado: solo se queda la muestra
    return {"ruta": f"{anio}/{carpeta.name}/video/{destino.name}", "min": dur / 60,
            "snr": float(np.median(snr)), "muestra": muestra}


LISTA = lista_de_episodios(EPISODIOS)
print(f"{len(LISTA)} episodio(s): {', '.join(LISTA)}")

In [ ]:
# 3. Hacer los vídeos, uno detrás de otro (si uno falla, se apunta y se sigue)
HECHOS, SALTADOS, FALLOS = {}, {}, {}
t_total = time.time()
for i, ep in enumerate(LISTA, 1):
    t0 = time.time()
    try:
        r = hacer_video(ep)
        HECHOS[ep] = r
        print(f"[{i}/{len(LISTA)}] {ep}  OK  {r['min']:.1f} min · audio {r['snr']:.0f} dB · {time.time() - t0:.0f} s  →  {r['ruta']}")
    except Salto as e:
        SALTADOS[ep] = str(e)
        print(f"[{i}/{len(LISTA)}] {ep}  saltado: {e}")
    except Exception as e:
        FALLOS[ep] = str(e)
        print(f"[{i}/{len(LISTA)}] {ep}  FALLO: {e}")

drive.flush_and_unmount()   # espera a que Drive termine de sincronizar: no cierres el cuaderno antes
print(f"\nHechos: {len(HECHOS)} · saltados: {len(SALTADOS)} · con fallo: {len(FALLOS)} · {(time.time() - t_total) / 60:.0f} min")
for ep, motivo in FALLOS.items():
    print(f"  FALLO {ep}: {motivo}")
if HECHOS:
    print("Siguiente: pide a la sesión de dirección el título y la descripción de cada uno, y súbelos en YouTube Studio.")

In [ ]:
# 4. Escuchar 30 s de cada vídeo hecho (a partir del minuto 1, sacados del .mp4 ya hecho)
from IPython.display import Audio, display
for ep, r in HECHOS.items():
    print(ep, "·", r["ruta"])
    display(Audio(str(r["muestra"])))